**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/fujisawa-2008"], check=True)
    os.chdir("/content/encore/entries/fujisawa-2008")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings checked on DANDI:000067 version 0.261106.2004 and CRCNS pfc-2 (unversioned)

This notebook belongs to the reconciliation of attempt fujisawa-2008-03 with attempt
fujisawa-2008-02 (docs/protocol.md, "Repeat attempts"). fujisawa-2008-02 used CRCNS pfc-2 only.
Of its twelve findings, seven (1, 3, 4, 5, 8, 10 and 12) are reported again by fujisawa-2008-03.
The other five, findings 2, 6, 7, 9 and 11, are checked here on the files as they are now:
CRCNS pfc-2 (doi:10.6080/K01V5BWK, unversioned, the bucket `crcnsarchive` under `pfc-2/live/`) and
its data description, version 0.7. Finding 9 is a question about the paper, not the files.
Each section opens with the earlier statement. The files are streamed with the entry's helper
module, `fujisawa.py`, which also pins DANDI:000067 at version 0.261106.2004; no NWB file is read
here, as none of the five findings is about them.

The conclusion for each finding is in the reconciliation block of `attempts/fujisawa-2008-03.yaml`.

In [ ]:
import io
import re
import tarfile
import time
import zipfile

import fsspec
import numpy as np
import requests
import scipy.io as sio
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

import fujisawa as fj

print("CRCNS:", fj.CRCNS, "| DANDI:", fj.DANDI_ID, fj.DANDI_VERSION)
t0 = time.time()
M = fj.load_crcns()   # behaviors.zip (12 sessions) and EE188_Behavior.mat / EE188_Behavior_PFC.mat
print(f"summary files read in {time.time() - t0:.0f} s")

# One pass over EE.188.tar.gz: the list of members with the compressed offset reached at each,
# EE188_example.mat, and channels 97 and 98 (the nose-poke sensors of rat EE, data description,
# page 1) of the 104-channel, 1,250 Hz .eeg file.
t0 = time.time()
TAR, EXAMPLE_OFFSET, EX, SENS = [], None, None, None
with fsspec.open(fj.CRCNS + "data/EE0708/EE.188.tar.gz", block_size=2**24).open() as fh:
    tf = tarfile.open(fileobj=fh, mode="r|gz")
    for m in tf:
        TAR.append((m.name, m.size, fh.tell()))
        if m.name.endswith("EE188_example.mat"):
            EXAMPLE_OFFSET = fh.tell()
            EX = sio.loadmat(io.BytesIO(tf.extractfile(m).read()))
        elif m.name.endswith(".eeg"):
            fe, parts = tf.extractfile(m), []
            while b := fe.read(208 * 100000):
                parts.append(np.frombuffer(b, np.int16).reshape(-1, 104)[:, 96:98].copy())
            SENS = np.concatenate(parts)
print(f"EE.188.tar.gz streamed in {time.time() - t0:.0f} s")

In [ ]:
for k in fj.SESSIONS:
    m = M[k]
    print(f"{k}: {len(m['CellsInfo'])} cells, {len(m['spiket'].ravel())} spikes, {len(m['SessionNP'])} correct trials "
          f"(left {(m['SessionNP'][:, 3] == 2).sum()}, right {(m['SessionNP'][:, 3] == 1).sum()}), "
          f"{len(m['whlrld'])} position frames ({len(m['whlrld']) / fj.WHL_RATE:.0f} s); spikeX {m['spikeX'].min():.3f} to "
          f"{m['spikeX'].max():.3f}; spikeLtr up to {int(m['spikeLtr'].max())}, spikeRtr up to {int(m['spikeRtr'].max())}")
print(f"EE.188.tar.gz: {len(TAR)} members, {TAR[-1][2]} compressed bytes read")
print("EE188_example.mat variables:", [k for k in EX if not k.startswith("__")], "| CellsInfo", EX["CellsInfo"].shape,
      "| pairs_all", EX["pairs_all"].shape)
print(f"EE.188.eeg channels 97 and 98: {SENS.shape[0]} samples ({SENS.shape[0] / 1250:.1f} s at 1,250 Hz), "
      f"ranges {SENS[:, 0].min()} to {SENS[:, 0].max()} and {SENS[:, 1].min()} to {SENS[:, 1].max()}")
print("Check: the .eeg file and the position file of EE188 cover the same time:",
      f"{SENS.shape[0] / 1250:.1f} s and {len(M['EE188']['whlrld']) / fj.WHL_RATE:.1f} s")

## Earlier finding 2 (metadata_error)

> The data description names EE188_example.mat as a file of folder EE0708; it is a member of
> EE0708/EE.188.tar.gz, about 2 GB into the gzip stream, and the folder holds EE188_Behavior.mat,
> EE188_Behavior_PFC.mat and EE188_Behavior_CA1.mat instead.

The folder's listing in the bucket and in the data set's `filelist.txt`, the description's
passages on EE188_example.mat and on the session tars (from the .docx of the description, version
0.7), and where in the tar the member lies.

In [ ]:
r = requests.get("https://crcnsarchive.s3-us-west-2.amazonaws.com/",
                 params={"list-type": "2", "prefix": "pfc-2/live/data/EE0708/"}, timeout=60)
sizes = {k.split("/")[-1]: int(z) for k, z in re.findall(r"<Key>([^<]*)</Key>.*?<Size>(\d+)</Size>", r.text) if not k.endswith("/")}
keys = list(sizes)
print("bucket folder EE0708:", keys)
print("  of these, named EE188*:", [k for k in keys if k.startswith("EE188")],
      "| EE188_example.mat present:", "EE188_example.mat" in keys)
fl = requests.get(fj.CRCNS + "filelist.txt", timeout=60).text
print("filelist.txt lines naming EE188_:", [l.strip() for l in fl.splitlines() if "EE188_" in l])
print("filelist.txt names example:", "example" in fl)

docx = requests.get(fj.CRCNS + "docs/crcns_pfc-2_data_description.docx", timeout=60).content
xml = zipfile.ZipFile(io.BytesIO(docx)).read("word/document.xml").decode()
paras = [re.sub(r"<[^>]+>", "", p) for p in re.findall(r"<w:p[ >].*?</w:p>", xml)]
for p in paras:
    if "EE188_example" in p or p.startswith("Data recorded on each day") or p.startswith("Version 0.7"):
        print("description:", p)

name, size, off = [x for x in TAR if x[0].endswith("EE188_example.mat")][0]
print(f"member {name} ({size} bytes) is reached {EXAMPLE_OFFSET / 1e9:.2f} GB into EE.188.tar.gz "
      f"({sizes['EE.188.tar.gz'] / 1e9:.2f} GB), member {[x[0] for x in TAR].index(name) + 1} of {len(TAR)}")

The cell above shows whether the file is still only inside the session tar, and what the
description says of it.

## Earlier finding 6 (metadata_error)

> The spikes' trial labels (spikeLtr, spikeRtr) and positions (spikeX) start at the first
> nose-poke of each trial, while the description defines the trial start as the end of the last
> nose-poke (SessionNP column 2). Spikes during the nose-poke, which lasts up to 315 s, are placed
> near position 0.

For every session's `_Behavior_PFC.mat`: the spikes carrying a left or right trial number, taken
as the n-th correct trial of that side in `SessionNP`, and how many fall between that trial's
first nose-poke (column 1) and its start (column 2), before the first poke, or after the trial
end; and the `spikeX` of the spikes in the nose-poke.

In [ ]:
rows = []
for k in fj.SESSIONS:
    m = M[k]
    t = m["spiket"].ravel() / fj.FS
    X = m["spikeX"].ravel()
    NP = m["SessionNP"]
    lab_n = poke_n = before = after = 0
    xs = []
    for side, lab in ((2, m["spikeLtr"].ravel()), (1, m["spikeRtr"].ravel())):
        tr = NP[NP[:, 3] == side]
        sel = lab > 0
        j = lab[sel].astype(int) - 1
        tt = t[sel]
        a, b, c = tr[j, 0], tr[j, 1], tr[j, 2]
        p = (tt >= a) & (tt < b)
        lab_n += sel.sum()
        poke_n += p.sum()
        before += (tt < a).sum()
        after += (tt > c).sum()
        xs.append(X[sel][p])
    xs = np.concatenate(xs)
    rows.append((k, lab_n, poke_n, before, after, xs, (NP[:, 1] - NP[:, 0]).max()))
    print(f"{k}: {lab_n} spikes with a trial label; {poke_n} ({100 * poke_n / lab_n:.1f}%) between the first poke and the "
          f"trial start, {before} before the first poke, {after} after the trial end; their spikeX median "
          f"{np.median(xs):.3f}, 95th percentile {np.percentile(xs, 95):.3f}, range {xs.min():.3f} to {xs.max():.3f}; "
          f"longest first poke to trial start {(M[k]['SessionNP'][:, 1] - M[k]['SessionNP'][:, 0]).max():.1f} s")
print(f"all sessions: {sum(r[2] for r in rows)} of {sum(r[1] for r in rows)} labelled spikes in the nose-poke; "
      f"sessions with any: {sum(r[2] > 0 for r in rows)} of {len(rows)}; longest first poke to trial start "
      f"{max(r[6] for r in rows):.1f} s")
for p in paras:
    if p.strip().startswith(("2st column", "[3] spikeX", "[4] spikeRtr", "[5] spikeLtr")):
        print("description:", p.strip())

The cell above shows whether the labelled spikes still include the nose-poke, and where their
positions lie.

## Earlier finding 7 (metadata_insufficient)

> The pairs_all list of EE188_example.mat (43 connections) is described as the cell pairs with
> monosynaptic connections, but it holds fewer connections than the paper reports for that session
> (48 and 30 in Figure 3b) and its largest component is the 39-cell circuit of Figure 3d; which
> selection it is, is not said.

The number of connections in `pairs_all`, the largest connected component of the undirected graph
they form over the file's 117 cells, the presynaptic cells' types in `CellsInfo`, and the
description's text for `pairs_all`. The paper's 48 and 30 (Figure 3b) and 39 of 117 (Figure 3d)
are those of the claims file, claims `fig3b-session-connections` and `fig3d-hub`.

In [ ]:
pa = EX["pairs_all"].astype(int)
cells = EX["CellsInfo"][:, 0].astype(int)
ctype = EX["CellsInfo"][:, 2].astype(int)
idx = {c: i for i, c in enumerate(cells)}
n = len(cells)
A = coo_matrix((np.ones(len(pa)), ([idx[a] for a in pa[:, 0]], [idx[b] for b in pa[:, 1]])), shape=(n, n))
ncomp, comp = connected_components(A, directed=True, connection="weak")
in_pairs = sorted(set(pa[:, 0]) | set(pa[:, 1]))
print(f"pairs_all: {len(pa)} rows, {len({tuple(r) for r in pa})} distinct directed pairs, over {len(in_pairs)} of {n} cells")
print(f"largest connected component: {np.bincount(comp).max()} cells")
print("presynaptic cell types (CellsInfo: pyramidal, interneuron, unknown):",
      np.bincount(ctype[[idx[a] for a in pa[:, 0]]], minlength=4)[1:].tolist(),
      "| connections from pyramidal and interneuron references against the paper's 48 and 30:",
      int((ctype[[idx[a] for a in pa[:, 0]]] == 1).sum()), int((ctype[[idx[a] for a in pa[:, 0]]] == 2).sum()))
i = [j for j, p in enumerate(paras) if p.strip().startswith("[8] pairs_all")][0]
print("description:", " / ".join(p.strip() for p in paras[i:i + 4] if p.strip()))

The cell above shows the size of the list and whether the description says how it was selected.

## Earlier finding 9 (methods_unavailable)

> Question for the paper's authors. Was the 99% global band of the jitter test the minimal band
> holding 99% of the surrogates entirely within its upper and lower sides, as the Methods' stepping
> procedure defines it (used here, G-b), or 1% on each side? With the stepping procedure this
> attempt finds 0.94 to 1.32% of directed pairs connected, with 1% on each side 1.07 to 1.50%,
> against 0.79%. It also asks which bin width the cross-correlograms used (1 ms or 0.5 ms) and how
> Figure 3d's subset of connections (pairs_all) was chosen.

This is a question about the paper's method, not about the files, so no dataset version can
settle it. The paper is the one both attempts read, and no answer from its authors is recorded.
The main notebook of this entry, `reproduce_fujisawa_2008.ipynb`, builds the global band from
the 99th percentile of the surrogates' maxima and the 1st percentile of their minima
(`fujisawa.classify_ccg`), that is 1% on each side, and runs both bin widths as readings B-a and
B-b (claim `fig3-connection-rates`). How `pairs_all` was chosen is checked under finding 7 above.

## Earlier finding 11 (selection_undocumented)

> The trial table of EE188 (SessionNP) leaves out 3 trials that the nose-poke sensor channels and
> the position file show as correct (first pokes at 822.9, 2245.1 and 2370.7 s), and 1 error trial
> (2687.2 s). In two of the three (822.9 and 2370.7 s) the rat reaches the arm 193.4 and 176.9 s
> after its last poke, longer than any table trial, so they may be left out for that reason; why any
> are left out is not said.

Every trial of EE188 found from the sensor channels and the position file, by the earlier
attempt's rule: pokes are low stretches of channel 97 (right) or 98 (left) below -15,000 (the
channels sit near 0 and drop to about -30,000 in a poke, a standard halfway threshold); a trial is
the run of pokes since the last arm-end visit, its type the channel poked, and its choice the arm
end reached next (within 60 pixels of the median position at the end of the table's correct trials
of that side; 40 and 80 pixels as variants). A detected trial is in the table when a table trial's
first poke lies between its first poke and the end of its last poke.

In [ ]:
m = M["EE188"]
NP, w = m["SessionNP"], m["whlrld"]


def poke_intervals(v, rate=1250.0, thr=-15000):
    on = v < thr
    d = np.diff(on.astype(int), prepend=0, append=0)
    return np.c_[np.flatnonzero(d == 1), np.flatnonzero(d == -1)] / rate


PK = {1: poke_intervals(SENS[:, 0]), 2: poke_intervals(SENS[:, 1])}
t_f = np.arange(len(w)) / fj.WHL_RATE
ok_f = w[:, 0] > 0
two = w[:, 2] > 0
px = np.where(two, (w[:, 0] + w[:, 2]) / 2, w[:, 0])
py = np.where(two, (w[:, 1] + w[:, 3]) / 2, w[:, 1])
ends = {}
for side in (1, 2):
    f = np.minimum((NP[NP[:, 3] == side, 2] * fj.WHL_RATE).astype(int), len(w) - 1)
    ends[side] = (np.median(px[f]), np.median(py[f]))
print(f"pokes detected: right channel {len(PK[1])}, left channel {len(PK[2])}; arm ends (pixels) right "
      f"{np.round(ends[1], 0)}, left {np.round(ends[2], 0)}")
print(f"table: {len(NP)} correct trials; first poke to trial end {(NP[:, 2] - NP[:, 0]).min():.1f} to "
      f"{(NP[:, 2] - NP[:, 0]).max():.1f} s, last poke to trial end up to {(NP[:, 2] - NP[:, 1]).max():.1f} s")


def all_trials(radius):
    ev = []
    for sd in (1, 2):
        near = ok_f & (np.hypot(px - ends[sd][0], py - ends[sd][1]) < radius)
        ev += [(t_f[i], "arm", sd, t_f[i]) for i in np.flatnonzero(np.diff(near.astype(int), prepend=0) == 1)]
        ev += [(a, "poke", sd, b) for a, b in PK[sd]]
    ev.sort()
    out, pending = [], []
    for t, kind, sd, t_end in ev:
        if kind == "poke":
            pending.append((t, sd, t_end))
        elif pending:
            # first poke, channels poked, arm reached, arrival, end of the last poke
            out.append((pending[0][0], {c for _, c, _ in pending}, sd, t, pending[-1][2]))
            pending = []
    return out


for radius in (60, 40, 80):
    tr = all_trials(radius)
    corr = [x for x in tr if len(x[1]) == 1 and min(x[1]) == x[2]]
    err = [x for x in tr if len(x[1]) == 1 and min(x[1]) != x[2]]
    left_out = [x for x in tr if not ((NP[:, 0] >= x[0] - 0.05) & (NP[:, 0] <= x[4] + 0.05)).any()]
    unmatched = sum(not any(x[0] - 0.05 <= r[0] <= x[4] + 0.05 for x in corr) for r in NP)
    print(f"radius {radius} px: {len(tr)} trials, {len(corr)} correct, {len(err)} error; table trials not within a detected "
          f"correct trial: {unmatched}; detected trials not in the table: {len(left_out)}")
    for x in left_out:
        kind = "correct" if x in corr else ("error" if x in err else "pokes on both channels")
        print(f"    first poke {x[0]:.1f} s, channel {sorted(x[1])}, arm {x[2]}, {kind}; arm reached {x[3] - x[0]:.1f} s "
              f"after the first poke, {x[3] - x[4]:.1f} s after the last")

The cell above shows which trials the table still leaves out. The description says only that
"Only correct trials are listed" in SessionNP (finding 6 above prints its column text).